# 🧬 NP-DNA Nano — Self-Contained Colab Training

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**What this does:**
1. Mounts Google Drive
2. Clones NP-DNA source from Drive
3. Installs dependencies (torch, etc.)
4. Downloads model checkpoint + dataset from Drive
5. Trains with auto-resume
6. Uploads checkpoint + metrics back to Drive

No SSH, no controller, no multi-worker — just **Run All**.

In [ ]:
# ═══ ① Mount Google Drive ═══
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ═══ ② Install dependencies ═══
import subprocess, sys, os, shutil, json, time
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/npdna_training')

def run(cmd, **kw):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600, **kw)

# Python 3.11+ preferred; install torch with CUDA
PY_VER = sys.version_info
print(f'Python {PY_VER.major}.{PY_VER.minor}.{PY_VER.micro}')

if not shutil.which('nvcc'):
    run('pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118')
else:
    run('pip install -q torch torchvision torchaudio')
run('pip install -q numpy psutil')

import torch
print(f'✅ torch {torch.__version__}')
print(f'🖥️  CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ═══ ③ Clone source from Drive ═══
# Prefer uploaded local source so Colab runs the same fixes as this workspace.
MODEL_DIR = DRIVE_BASE / 'model' / 'nano'
DATA_DIR = DRIVE_BASE / 'data'

# Check Drive has everything
required = [MODEL_DIR / 'model.pt', MODEL_DIR / 'genome.pt', MODEL_DIR / 'tokenizer.json',
            MODEL_DIR / 'vocabulary.pt', MODEL_DIR / 'metadata.json']
for p in required:
    if not p.exists():
        print(f'⚠️  Missing: {p.name}')
for p in required:
    if p.exists():
        sz = p.stat().st_size / 1e6
        print(f'  ✓ {p.name} ({sz:.1f} MB)')

print()
SOURCE_DIR = DRIVE_BASE / 'source' / 'tantra'
if os.path.exists('/content/tantra'):
    shutil.rmtree('/content/tantra')
if SOURCE_DIR.exists():
    shutil.copytree(SOURCE_DIR, '/content/tantra')
    print('Loaded patched tantra source from Drive')
else:
    print('Drive source missing; falling back to GitHub source')
    run('git clone --depth 1 https://github.com/atulyaai/Atulya-Tantra /content/repo')
    shutil.copytree('/content/repo/tantra', '/content/tantra')
    shutil.rmtree('/content/repo', ignore_errors=True)

# Add to path
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

print('Source ready at /content/tantra')

In [ ]:
# ═══ ④ Copy model checkpoint + data to local ═══
LOCAL_MODEL = Path('/content/model')
LOCAL_DATA = Path('/content/data')
LOCAL_OUTPUT = Path('/content/output')

LOCAL_MODEL.mkdir(exist_ok=True)
LOCAL_DATA.mkdir(exist_ok=True)
LOCAL_OUTPUT.mkdir(exist_ok=True)

# Copy model files
print('📥 Copying model...')
for f in MODEL_DIR.iterdir():
    dst = LOCAL_MODEL / f.name
    # Always overwrite — don't rely on stale cache
    if dst.is_dir():
        shutil.copytree(f, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(f, dst)
print(f'  ✓ {len(list(LOCAL_MODEL.iterdir()))} files ({sum(f.stat().st_size for f in LOCAL_MODEL.iterdir() if f.is_file())/1e6:.1f} MB)')
# Copy dataset
print('📥 Copying dataset...')
DATASET_FILE = 'all_datasets.jsonl'
src_data = DATA_DIR / DATASET_FILE
dst_data = LOCAL_DATA / DATASET_FILE
if src_data.exists():
    shutil.copy2(src_data, dst_data)
    print(f'  ✓ {DATASET_FILE} ({src_data.stat().st_size/1e6:.1f} MB)')
else:
    print(f'  ✓ Not found: {DATASET_FILE}')

# Verify
print(f'\n📊 Local setup:')
print(f'  Model: {LOCAL_MODEL}')
print(f'  Data:  {LOCAL_DATA / DATASET_FILE}')
print(f'  Output: {LOCAL_OUTPUT}')

In [ ]:
# ═══ ⑤ Train NP-DNA with auto-resume ═══

STEPS = 5000           # Total training steps
BATCH_SIZE = 4         # Per-device batch size
SEQ_LIMIT = 256        # Context length
LR = 2e-3              # Learning rate
CONFIG_NAME = 'nano'   # Model config: nano/micro/small/atulya_seed

# Resume from the latest Drive model unless a newer in-session checkpoint exists.
RESUME_PATH = str(LOCAL_MODEL) if (LOCAL_MODEL / 'model.pt').exists() else None
checkpoint_dir = LOCAL_OUTPUT / 'checkpoints'
if checkpoint_dir.exists():
    step_dirs = []
    for d in checkpoint_dir.iterdir():
        if d.is_dir() and d.name.startswith('step_'):
            try:
                step = int(d.name.split('_')[1])
                step_dirs.append((step, d))
            except (ValueError, IndexError):
                pass
    if step_dirs:
        step_dirs.sort(key=lambda x: x[0])
        RESUME_PATH = str(step_dirs[-1][1])
        print(f'Resuming from in-session checkpoint: step_{step_dirs[-1][0]}')
if RESUME_PATH == str(LOCAL_MODEL):
    print(f'Resuming from Drive model: {LOCAL_MODEL}')
elif RESUME_PATH is None:
    print('Starting fresh training')

print(f'\n🏋️  Training config: {CONFIG_NAME}, {STEPS} steps, lr={LR}, batch={BATCH_SIZE}')
print(f'   seq_limit={SEQ_LIMIT}, data={str(LOCAL_DATA / DATASET_FILE).split("/")[-1]}')

from tantra.training.npdna_train import train_npdna

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'\n🔧 Device: {device} ({gpu_name})')

start = time.time()
core, losses = train_npdna(
    config_name=CONFIG_NAME,
    max_steps=STEPS,
    lr=LR,
    batch_size=BATCH_SIZE,
    seq_limit=SEQ_LIMIT,
    data_path=str(LOCAL_DATA / DATASET_FILE),
    log_every=50,
    checkpoint_every=250,
    output_dir=str(LOCAL_OUTPUT),
    resume_from=RESUME_PATH,
    device=device,
    bf16=(device == 'cuda'),
    pack_sequences=True,
)
elapsed = time.time() - start
final_loss = float(losses[-1]) if losses else None
loss_text = f'{final_loss:.4f}' if final_loss is not None else 'N/A'
print(f'\n✅ Training complete: {elapsed/60:.1f} min, final loss: {loss_text}')

In [ ]:
# ═══ ⑥ Upload results to Drive ═══
import shutil, json
from datetime import datetime

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'colab_run_{TIMESTAMP}'

# Destination on Drive
DRIVE_RUN_DIR = DRIVE_BASE / 'colab_runs' / RUN_NAME
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'📤 Uploading results to Drive: {DRIVE_RUN_DIR}')

# Upload output files
uploaded = 0
for f in LOCAL_OUTPUT.iterdir():
    if f.is_file() and f.suffix in ('.pt', '.json', '.jsonl', '.log'):
        shutil.copy2(f, DRIVE_RUN_DIR / f.name)
        sz = f.stat().st_size / 1e6
        print(f'  ✓ {f.name} ({sz:.1f} MB)')
        uploaded += 1

# Upload checkpoints (last 3)
ckpt_src = LOCAL_OUTPUT / 'checkpoints'
ckpt_dst = DRIVE_RUN_DIR / 'checkpoints'
if ckpt_src.exists():
    step_dirs = []
    for d in ckpt_src.iterdir():
        if d.is_dir() and d.name.startswith('step_'):
            try:
                step = int(d.name.split('_')[1])
                step_dirs.append((step, d))
            except (ValueError, IndexError):
                pass
    step_dirs.sort(key=lambda x: x[0])
    for _, d in step_dirs[-3:]:  # Last 3 checkpoints
        shutil.copytree(d, ckpt_dst / d.name, dirs_exist_ok=True)
        print(f'  ✓ checkpoint/{d.name}')

# Update the complete resumable latest checkpoint on Drive.
LATEST_DIR = DRIVE_BASE / 'model' / 'nano'
print(f'\n🔄 Updating latest model on Drive...')
for name in ('model.pt', 'metadata.json', 'tokenizer.json'):
    src = LOCAL_OUTPUT / name
    if src.exists():
        shutil.copy2(src, LATEST_DIR / name)
        print(f'  model update: {name}')
if (LOCAL_OUTPUT / 'cortex').exists():
    shutil.copytree(LOCAL_OUTPUT / 'cortex', LATEST_DIR / 'cortex', dirs_exist_ok=True)
    print('  model update: cortex/')

print(f'\n✅ Upload complete: {uploaded} files + checkpoints')
print(f'\n📋 Run summary:')
print(f'  Steps: {STEPS}')
print(f'  Config: {CONFIG_NAME}')
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  Final loss: {loss_text}')
print(f'  Time: {elapsed/60:.1f} min')
print(f'  Saved to: {DRIVE_RUN_DIR}')

In [ ]:
# ═══ ⑦ Keep alive (optional) ═══
# Colab would auto-disconnect after idle. This cell keeps the
# session alive for Drive sync. Comment out to let session end.
print('⏳ Keeping session alive for 5 min so Drive syncs...')
time.sleep(300)
print('✅ Done — you can close this session.')